In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import Adam
from torchinfo import summary

import torchvision
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

In [2]:
device = torch.device('mps') if torch.backends.mps.is_available() else 'cpu'
SEED = 1984
torch.manual_seed(SEED)
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 30

In [3]:
PATH = Path('.').absolute().parent
data_dir = PATH / 'data' / 'fruits-360_dataset_100x100'
train_data_dir = data_dir / 'Training'
test_data_dir  = data_dir / 'Test'

In [4]:
print('Data Directory:')
for files in data_dir.iterdir():
    print(f' ++ {files.name}')

classes =  [c.name for c in train_data_dir.iterdir()]
print(f'There are {len(classes)} classes : {classes}')

Data Directory:
 ++ .DS_Store
 ++ LICENSE
 ++ Test
 ++ Training
 ++ readme.md
There are 141 classes : ['Tomato 4', 'Corn Husk 1', 'Huckleberry 1', 'Tomato 3', 'Strawberry Wedge 1', 'Physalis 1', 'Pineapple 1', 'Avocado 1', 'Pear Kaiser 1', 'Grape Blue 1', 'Cactus fruit 1', 'Apple Granny Smith 1', 'Cherry 1', 'Tomato 2', 'Grapefruit Pink 1', 'Melon Piel de Sapo 1', 'Eggplant long 1', 'Grape White 1', 'Carrot 1', 'Redcurrant 1', 'Pear Stone 1', 'Maracuja 1', 'Nut Pecan 1', 'Quince 1', 'Nut Forest 1', 'Cocos 1', 'Grapefruit White 1', 'Raspberry 1', 'Apple Braeburn 1', 'Tamarillo 1', 'Banana Lady Finger 1', 'Hazelnut 1', 'Cabbage white 1', 'Mandarine 1', 'Kumquats 1', 'Apricot 1', 'Banana Red 1', 'Papaya 1', 'Mangostan 1', 'Apple Golden 2', 'Carambula 1', 'Peach Flat 1', 'Apple Red 1', 'Peach 1', 'Cherry Wax Yellow 1', 'Granadilla 1', 'Avocado ripe 1', 'Pomegranate 1', 'Mulberry 1', 'Mango 1', 'Apple Golden 3', 'Cucumber 3', 'Zucchini 1', 'Walnut 1', 'Tomato Heart 1', 'Cherry Rainier 1', '

In [5]:
dataset = ImageFolder(str(train_data_dir), transform=ToTensor())
testset = ImageFolder(str(test_data_dir),  transform=ToTensor())

print(f'Size of training dataset: {len(dataset)}')
print(f'Size of test dataset:     {len(testset)}')

Size of training dataset: 70491
Size of test dataset:     23619


In [6]:
valid_size = len(dataset) // 5
train_size = len(dataset) - valid_size

train_ds, valid_ds = random_split(dataset, [train_size, valid_size])
len(train_ds), len(valid_ds) # train_ds length = dataset length - val_ds length

(56393, 14098)

In [26]:
# modified from book example

def conv_layer(input_size, output_size, kernel_size, stride=1):
    return nn.Sequential(
        nn.Conv2d(input_size, output_size, kernel_size=kernel_size, stride=stride),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

def get_model():
    model = nn.Sequential(
        conv_layer(3, 32, 3),
        conv_layer(32, 64, 3),
        conv_layer(64, 128, 3),
        conv_layer(128, 256, 3),
        nn.Flatten(),
        nn.Linear(4096, 141),
        nn.Softmax(dim=1)
    ).to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)

    return model, loss_fn, optimizer

In [27]:
model, loss_fn, optimizer = get_model()
summary(model, input_size=(BATCH_SIZE, 3, 100, 100))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [32, 141]                 --
├─Sequential: 1-1                        [32, 32, 49, 49]          --
│    └─Conv2d: 2-1                       [32, 32, 98, 98]          896
│    └─ReLU: 2-2                         [32, 32, 98, 98]          --
│    └─MaxPool2d: 2-3                    [32, 32, 49, 49]          --
├─Sequential: 1-2                        [32, 64, 23, 23]          --
│    └─Conv2d: 2-4                       [32, 64, 47, 47]          18,496
│    └─ReLU: 2-5                         [32, 64, 47, 47]          --
│    └─MaxPool2d: 2-6                    [32, 64, 23, 23]          --
├─Sequential: 1-3                        [32, 128, 10, 10]         --
│    └─Conv2d: 2-7                       [32, 128, 21, 21]         73,856
│    └─ReLU: 2-8                         [32, 128, 21, 21]         --
│    └─MaxPool2d: 2-9                    [32, 128, 10, 10]         --
├─Sequ

In [7]:
def get_data(batch_size):
    train_loader = DataLoader(train_ds, batch_size, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size)
    test_loader = DataLoader(testset, batch_size)
    return train_loader, valid_loader, test_loader

In [ ]:
def train_batch(X, y, model, optimizer, loss_fn):
    model.train()
    prediction = model(X)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    return batch_loss.item()

@torch.no_grad()
def val_loss(X, y, model):
    model.eval()
    prediction = model(X)
    val_loss = loss_fn(prediction, y)
    return val_loss.item()

In [31]:
@torch.no_grad()
def accuracy(X, y, model):
    model.eval()
    prediction = model(X)
    max_values, argmaxes = prediction.max(-1)
    is_correct = argmaxes == y
    return is_correct.cpu().numpy().tolist()

In [32]:
trn_dl, val_dl = get_data(BATCH_SIZE)
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
valid_losses, valid_accuracies = [], []

temp_epochs = 5

for epoch in range(temp_epochs):
    print(epoch)

    train_epoch_losses, train_epoch_accuracies = [], []
    for idx, batch in enumerate(iter(trn_dl)):
        X, y = batch
        batch_loss = train_batch(X, y, model, optimizer, loss_fn)
        train_epoch_losses(batch_loss)
    train_epoch_loss = np.array(train_epoch_losses).mean()

    for idx, batch in enumerate(iter(trn_dl)):
        X, y = batch
        is_correct = accuracy(X, y, model)
        train_epoch_accuracies.extend(is_correct)
    train_epoch_accuracy = np.mean(train_epoch_accuracies)

    for idx, batch in enumerate(iter(val_dl)):
        X, y = batch
        val_is_correct = accuracy(X, y, model)
        validation_loss = val_loss(X, y, model)
    val_epoch_accuracy = np.mean(val_is_correct)

    train_losses.append(train_epoch_loss)
    train_accuracies.append(train_epoch_accuracy)
    valid_losses.append(validation_loss)
    valid_accuracies.append(val_epoch_accuracy)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 8), dpi=200)
ax0.plot(np.arange(temp_epochs)+1, train_losses, label='Training loss')
ax0.plot(np.arange(temp_epochs)+1, valid_losses, label='Validation loss')
ax0.set_title('Training and validation loss', fontweight='bold')
ax0.legend()
ax1.plot(np.arange(temp_epochs)+1, train_accuracies, label='Training accuracy')
ax1.plot(np.arange(temp_epochs)+1, valid_accuracies, label='Validation accuracy')
ax1.set_title('Training and validation accuracies', fontweight='bold')
ax1.legend()

0


RuntimeError: Mismatched Tensor types in NNPack convolutionOutput

## Second attempt

This time using the `class` approach.

In [17]:
class CNN1(nn.Module):
    
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv2d1 = nn.Conv2d(3,   32,  kernel_size=3)
        self.conv2d2 = nn.Conv2d(32,  64,  kernel_size=3)
        self.conv2d3 = nn.Conv2d(64,  128, kernel_size=3)
        self.conv2d4 = nn.Conv2d(128, 256, kernel_size=3)
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()
        self.relu3 = nn.ReLU()
        self.relu4 = nn.ReLU()
        self.maxpool2d1 = nn.MaxPool2d(2)
        self.maxpool2d2 = nn.MaxPool2d(2)
        self.maxpool2d3 = nn.MaxPool2d(2)
        self.maxpool2d4 = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(4096, 141)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.conv2d1(x)
        x = self.relu1(x)
        x = self.maxpool2d1(x)
        x = self.conv2d2(x)
        x = self.relu2(x)
        x = self.maxpool2d2(x)
        x = self.conv2d3(x)
        x = self.relu3(x)
        x = self.maxpool2d3(x)
        x = self.conv2d4(x)
        x = self.relu4(x)
        x = self.maxpool2d4(x)
        x = self.flatten(x)
        x = self.linear(x)
        output = self.softmax(x)
        return output

In [18]:
model = CNN1()
summary(model, input_size=(BATCH_SIZE, 3, 100, 100))

Layer (type:depth-idx)                   Output Shape              Param #
CNN1                                     [32, 141]                 --
├─Conv2d: 1-1                            [32, 32, 98, 98]          896
├─ReLU: 1-2                              [32, 32, 98, 98]          --
├─MaxPool2d: 1-3                         [32, 32, 49, 49]          --
├─Conv2d: 1-4                            [32, 64, 47, 47]          18,496
├─ReLU: 1-5                              [32, 64, 47, 47]          --
├─MaxPool2d: 1-6                         [32, 64, 23, 23]          --
├─Conv2d: 1-7                            [32, 128, 21, 21]         73,856
├─ReLU: 1-8                              [32, 128, 21, 21]         --
├─MaxPool2d: 1-9                         [32, 128, 10, 10]         --
├─Conv2d: 1-10                           [32, 256, 8, 8]           295,168
├─ReLU: 1-11                             [32, 256, 8, 8]           --
├─MaxPool2d: 1-12                        [32, 256, 4, 4]           --
├

In [19]:
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

In [20]:
train_loader, valid_loader, test_loader = get_data(BATCH_SIZE)

This next part follows https://www.digitalocean.com/community/tutorials/writing-cnns-from-scratch-in-pytorch. 

In [21]:
model.to(device);

In [ ]:
for epoch in range(EPOCHS):

    for i, (images, labels) in enumerate(tqdm(train_loader)):  
        # Move tensors to the configured device
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images).to(device)
        loss = loss_fn(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {loss.item():.4f}')

In [ ]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images).to(device)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    print(f'Accuracy of the network on the train images: {100 * correct / total:.4f}%')

  0%|          | 0/1763 [00:00<?, ?it/s]

Accuracy of the network on the train images: 0.6614%


In [47]:
@torch.no_grad()
def evaluate(model, valid_loader):
    model.eval()
    outputs = [model.validation_step(batch) for batch in valid_loader]
    return model.validation_epoch_end(outputs)

def fit(epochs, lr, model, train_loader, valid_loader, opt_func=torch.optim.SGD):
    history = []
    optimizer = opt_func(model.parameters(), lr)
    for epoch in range(epochs):
        # Training Phase 
        model.train()
        train_losses = []
        for batch in tqdm(train_loader):
            loss = model.training_step(batch)
            train_losses.append(loss)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        # Validation phase
        result = evaluate(model, valid_loader)
        result['train_loss'] = torch.stack(train_losses).mean().item()
        model.epoch_end(epoch, result)
        history.append(result)
    return history